In [1]:
import sys as __harness_sys
from pathlib import Path as __harness_Path
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\scripts')
from _offline import enforce_offline as __guard, blocked_attempts as __blocked
__guard()
import capture_runresults_213 as __capture
__capture.install()
import agentic_systems as __harness_toolkit
assert __harness_toolkit.__version__ == "2.1.3"
assert "site-packages" in __harness_Path(__harness_toolkit.__file__).parts

# LangGraph: checkpoint y reanudación

Input: gasto de 280. Proceso: LangGraph interrumpe, guarda el estado y recibe una aprobación explícita. Output: paused=True, después approved=True. La Action conecta esa capacidad nativa con RunResult. No requiere un modelo.

In [2]:
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3"

In [3]:
from typing import TypedDict
from uuid import uuid4
import agentic_systems as toolkit

def run():
    from langgraph.graph import StateGraph, START, END
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.types import interrupt, Command

    class State(TypedDict, total=False):
        amount: int
        approved: bool

    def approval(state):
        decision = interrupt({'question': 'Approve this expense?', 'amount': state['amount']})
        if type(decision) is not bool:
            raise ValueError('An explicit boolean approval is required')
        return {'approved': decision}
    graph = StateGraph(State)
    graph.add_node('approval', approval)
    graph.add_edge(START, 'approval')
    graph.add_edge('approval', END)
    compiled = graph.compile(checkpointer=InMemorySaver())
    config = {'configurable': {'thread_id': 'expense-' + uuid4().hex}}

    def step(request):
        raw = compiled.invoke(Command(resume=request['resume']) if 'resume' in request else request, config=config)
        if raw.get('__interrupt__'):
            return {'paused': True, 'request': raw['__interrupt__'][0].value}
        return {'paused': False, 'amount': raw['amount'], 'approved': raw['approved']}
    actor = toolkit.Agent('human_approval', toolkit.Action('checkpoint', step))
    paused = actor.run({'amount': 280})
    resumed = actor.run({'resume': True})
    assert paused.ok and paused.data['paused']
    assert resumed.ok and resumed.data == {'paused': False, 'amount': 280, 'approved': True}
    return {'feature': 'LangGraph interrupt/checkpoint/resume', 'input': {'amount': 280}, 'paused': paused.to_dict(), 'resumed': resumed.to_dict(), 'inference_requests': 0, 'scope': 'Native LangGraph feature wrapped by Action; not an added toolkit constructor'}

In [4]:
report = run()
print({"input": report["input"], "paused": report["paused"]["data"], "output": report["resumed"]["data"]})

{'input': {'amount': 280}, 'paused': {'paused': True, 'request': {'question': 'Approve this expense?', 'amount': 280}}, 'output': {'paused': False, 'amount': 280, 'approved': True}}


In [5]:
__capture.save('C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\revisions\\2.1.3\\learning-evidence-elevated\\notebook-runresults\\12-01_langgraph.json', 'tutorials/frameworks/01_langgraph.ipynb')
assert __blocked() == 0